In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from langchain_openai import ChatOpenAI
from langchain_google_vertexai import ChatVertexAI

from wsd.load_data import load_data
from wsd.models import BinaryWSD, ClusterByMeaningModel, DummyComparator
from linpub.metrics import accuracy

In [3]:
X, y = load_data(lang='fr')

k = 200
X_test, y_test = X[:k], y[:k]

In [10]:
len(X), len(X_test)

(24266, 200)

In [4]:
from linalgo.annotate.models import Corpus, Document, Annotation, Target, Selector
from lineval.utils import Body
from datetime import datetime
from collections import defaultdict

In [8]:
Semcor_corpus = Corpus(name='Semcor')

grouped_X = defaultdict(list)
for row in X:
    grouped_X[(row.lemma, row.pos)].append(row)
items = grouped_X.items()
docs = []
for g, annos in items:
    contexts = "\n".join([anno.context for anno in annos])
    doc = Document(content=contexts,
                   corpus=Semcor_corpus
                   )
    doc_annos = []
    for a in annos:
        anno = Annotation(document=doc,
                          body=Body(text=a.text, context=a.context),
                          task="task",
                          annotator="gold_annotator",
                          target={},
                          created=datetime.now())
        doc_annos.append(anno)
    doc.annotations = set(doc_annos)
    docs.append(doc)

Semcor_corpus.documents = docs

In [9]:
len(Semcor_corpus.documents)

3581

In [9]:
X[0]

Candidate(pos='NOUN', text='objectifs', lemma='objectif', lemma_meaning='NA', example='Depuis combien de temps avez -vous revu les objectifs de votre programme de prestations et de services ?', context='Depuis combien de temps avez -vous revu les objectifs de votre programme de prestations et de services ?', document='NA')

In [ ]:
gpt = ChatOpenAI(temperature=0, model="gpt-4o-mini")\
.with_structured_output(BinaryWSD)
model1 = ClusterByMeaningModel(comparator=gpt)

gemini = ChatVertexAI(temperature=0, model="gemini-1.5-flash")\
    .with_structured_output(BinaryWSD)
model2 = ClusterByMeaningModel(comparator=gemini)

dummy = DummyComparator(probability=1)
model3 = ClusterByMeaningModel(comparator=dummy)

dummy = DummyComparator(probability=0)
model4 = ClusterByMeaningModel(comparator=dummy)

y_pred1 = model1.predict(X_test, verbose=True)
y_pred2 = model2.predict(X_test, verbose=True)
y_pred3 = model3.predict(X_test, verbose=True)
y_pred4 = model4.predict(X_test, verbose=True)

print(f"gpt-4o-mini:  {accuracy(y_pred1, y_test)}")
print(f"gemini1.5-flash: {accuracy(y_pred2, y_test)}")
print(f"dummy (always 1): {accuracy(y_pred3, y_test)}")
print(f"dummy (always 0): {accuracy(y_pred4, y_test)}")

In [ ]:
import pandas as pd

records = []
for yt, yp, x in zip(y_test, y_pred1, X_test):
    record = {'lemma': x.lemma, 'pos': x.pos, 'y': yt, 'y_pred': yp, 'text': x.text, 'context': x.context}
    records.append(record)
pd.DataFrame(records).sort_values(['lemma', 'pos']).to_csv('plop.csv', index=False)